In [ ]:
##Dependancies

from __future__ import annotations

import argparse
import copy
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
def _start_dirs() -> list[Path]:
    """Places to begin an upward search.

    `__file__` is missing (or misleading) inside Jupyter, so the current working
    directory has to be a first-class starting point — otherwise every path in
    this module silently resolves somewhere else when run from a notebook.
    """
    dirs = [Path.cwd().resolve()]
    try:
        dirs.append(Path(__file__).resolve().parent)
    except NameError:  # notebook / REPL
        pass
    return dirs


def _walk_up(start: Path):
    return [start, *start.parents]


def find_repo_root() -> Path:
    """Nearest ancestor holding CLAUDE.md — the repo root marker."""
    for start in _start_dirs():
        for directory in _walk_up(start):
            if (directory / "CLAUDE.md").is_file():
                return directory
    return Path.cwd().resolve()


REPO_ROOT = find_repo_root()


In [ ]:
# The dataset folder gets named inconsistently depending on where it came from
# ("UCI-HAR Dataset" from some mirrors, "UCI HAR Dataset" from the UCI zip), and
# it may sit at the repo root or under data/raw/.
DATASET_DIR_NAMES = ["UCI-HAR Dataset", "UCI HAR Dataset"]
DATASET_SUBPATHS = [Path("."), Path("data") / "raw"]


def _dataset_candidates() -> list[Path]:
    seen, out = set(), []
    roots = [REPO_ROOT, *_start_dirs()]
    for root in roots:
        for directory in _walk_up(root.resolve()):
            for sub in DATASET_SUBPATHS:
                for name in DATASET_DIR_NAMES:
                    candidate = (directory / sub / name).resolve()
                    if candidate not in seen:
                        seen.add(candidate)
                        out.append(candidate)
    return out


def is_dataset_root(path: Path) -> bool:
    return (path / "train" / "Inertial Signals").is_dir()


def find_dataset_root() -> Path:
    """First candidate that actually contains the inertial signals."""
    for candidate in _dataset_candidates():
        if is_dataset_root(candidate):
            return candidate
    return REPO_ROOT / DATASET_DIR_NAMES[0]  # for the error message in load_split


DEFAULT_ROOT = find_dataset_root()

# Parsed .npy cache. Lives under data/processed/ because .gitignore already
# excludes that tree — the arrays are ~46 MB.
CACHE_DIR = REPO_ROOT / "data" / "processed" / "uci_har"

# The 9 channels, in the conventional order. This fixes the meaning of the
# last input axis — keep it stable or trained weights stop matching the data.
CHANNELS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z",
]

WINDOW = 128
N_CHANNELS = len(CHANNELS)
N_CLASSES = 6

CLASS_NAMES = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING", ]

In [ ]:
def load_split(root, split, use_cache):
    #root --> main dataset directory path
    #split --> either "test" or "train"
    #use_cache --> use yes or no bool
    x_cache = CACHE_DIR / f"X_{split}.npy"
    y_cache = CACHE_DIR / f"Y_{split}.npy"

    if use_cache and x_cache.exists() and y_cache.exists():
        return np.load(x_cache), np.load(y_cache)

    signals_dir = root / split / "Internal Signals" #root-->split-->internal signals
    if not signals_dir.is_dir():
        looked = "\n".join(str(c) for c in _dataset_candidate()[:8])
        raise FileNotFound Error(
              f"{signals_dir} not found.\nSearched (first few):\n  {looked}\n"
               "Pass root= / --root pointing at the folder that contains train/ and test/."
        )
    channels = [
        np.loadtxt(signal_dir / f"{name}_{split}.txt", dtype=np.float32) for name in CHANNELS
    ]
    x = np.stack(channels, axis=-1) #stores the channels in a -1 axis stack 

    y = np.load.txt(root / split / f"y_{split}.txt", dtype=np.int64) - 1

    if use_cache:
        CACHE_DIR.mkdir(parents=True, exist_ok = True)
        np.save(x_cache, x)
        np.save(y_cache, y)

    return x, y

def make_loader(x: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y)) #Converts to a PyTorch Tensor because it has been used in numpy it needs to be converted
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    
        

    

In [ ]:
def init_lstm(lstm: nn.LSTM):
    hidden = lstm.hidden_size
    for name, param in lstm.named_paramters():
        if name.startswith("weight_ih"):
            nn.init.xavier_uniform_(param)
        elif name.startswith("weight_hh"):
            nn.init.orthogonal_(param)
        elif name.startswith("bias"):
            nn.init.zeroes_(param)
    with torch.no_grad():
        lstm.bias_ih_l0[hidden : 2 * hidden].fill_(1.0)